In [1]:
import requests # [+] HTTP 요청
from bs4 import BeautifulSoup  # [+] 웹스크래핑
import pandas as pd

In [2]:
# 1. User-Agent 설정
# 우리가 실제 크롬 브라우저를 사용하는 일반 사용자인 것처럼 서버를 속이는 '위장 신분증' 역할
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

search_keyword = 'lck'  # [+] 검색 키워드
url = f'https://search.naver.com/search.naver?where=news&query={search_keyword}'  # 검색한 웹페이지 주소

In [3]:
# 2. 페이지 요청 및 파싱
res = requests.get(url, headers)  # [+] 네이버 서버에 페이지 정보 요청(GET)
res.raise_for_status() # 서버가 응답을 잘 주었는지 확인(정상 200, 페이지 없음 404 등)
soup = BeautifulSoup(res.text, 'html.parser') # [+] BeautifulSoup 객체 생성

In [4]:
# 3. 질문자님이 찾은 클래스로 타겟팅 (띄어쓰기를 '.'으로 변경)
# [+] 찾고자 하는 HTML 태그 정의
target_class = 'span.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-1.sds-comps-text-type-headline1'
title_spans = soup.select(target_class)

extracted_data = []

In [5]:
title_spans

[<span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1"><mark>LCK</mark> 3주차, 젠지 꺾은 디플러스 기아의 상승세 유지 ‘주목’</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">젠지 잡은 기세 잇는다… DK, 3주 차서 KT와 맞대결 [<mark>LCK</mark>]</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">디플러스 기아, 5년만에 젠지 꺾고 ‘체증’ 풀었다[<mark>LCK</mark>]</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">[여전사 풍향계] 우리카드, <mark>LCK</mark> 시즌 맞아 고객 기반 확대 모색 外</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">[<mark>LCK</mark>] 3연패 후 첫 승 거둔 BNK 피어엑스</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">[<mark>LCK</mark> 톡톡] '비디디' 곽보성, "KT, 아직 발전할 가능성 높아 끝까지 좋은.

In [6]:
# 4. 데이터 추출
for span in title_spans:
    # [+] 기사 제목 추출
    title = span.text.strip()
    
    # [+] 기사 링크 추출: 제목(span)을 클릭 가능하게 만드는 부모 <a> 태그를 역추적하여 찾음
    parent_a_tag = span.find_parent('a')
    link = parent_a_tag['href'] if parent_a_tag and parent_a_tag.has_attr('href') else "링크 없음"
    
    extracted_data.append({
        '제목': title,
        '링크': link
    })

In [7]:
extracted_data

[{'제목': 'LCK 3주차, 젠지 꺾은 디플러스 기아의 상승세 유지 ‘주목’',
  '링크': 'https://www.mk.co.kr/article/12016220'},
 {'제목': '젠지 잡은 기세 잇는다… DK, 3주 차서 KT와 맞대결 [LCK]',
  '링크': 'https://www.xportsnews.com/article/2136326'},
 {'제목': '디플러스 기아, 5년만에 젠지 꺾고 ‘체증’ 풀었다[LCK]',
  '링크': 'https://sports.khan.co.kr/article/202604131304003?pt=nv'},
 {'제목': '[여전사 풍향계] 우리카드, LCK 시즌 맞아 고객 기반 확대 모색 外',
  '링크': 'https://www.ekn.kr/web/view.php?key=20260414020566854'},
 {'제목': '[LCK] 3연패 후 첫 승 거둔 BNK 피어엑스',
  '링크': 'https://www.inven.co.kr/webzine/news/?news=315341'},
 {'제목': '[LCK 톡톡] \'비디디\' 곽보성, "KT, 아직 발전할 가능성 높아 끝까지 좋은...',
  '링크': 'http://www.osen.co.kr/article/G1112780807'},
 {'제목': "[LCK] BFX, '빅라', '데이스타' 교체 기용 암시",
  '링크': 'https://www.inven.co.kr/webzine/news/?news=315343'},
 {'제목': "[기업家] 한진그룹 ㅣ 물류·항공 위기 대응부터 신사업 확장까지…'변...",
  '링크': 'https://www.cbci.co.kr/news/articleView.html?idxno=567426'},
 {'제목': '[Game & Now] 스마일게이트·LCK·SOOP, 콘텐츠 전방위 확장',
  '링크': 'https://www.ebn.co.kr/news/articleView.html?idxno=1705244'},
 

In [11]:
# 5. [+] DataFrame으로 변환 및 확인
df = pd.DataFrame(extracted_data)
print(f"총 {len(df)}개의 기사를 추출했습니다.\n")
display(df)

총 10개의 기사를 추출했습니다.



,제목,링크
0,"LCK 3주차, 젠지 꺾은 디플러스 기아의 상승세 유지 ‘주목’",https://www.mk.co.kr/article/12016220
1,"젠지 잡은 기세 잇는다… DK, 3주 차서 KT와 맞대결 [LCK]",https://www.xportsnews.com/article/2136326
2,"디플러스 기아, 5년만에 젠지 꺾고 ‘체증’ 풀었다[LCK]",https://sports.khan.co.kr/article/202604131304...
3,"[여전사 풍향계] 우리카드, LCK 시즌 맞아 고객 기반 확대 모색 外",https://www.ekn.kr/web/view.php?key=2026041402...
4,[LCK] 3연패 후 첫 승 거둔 BNK 피어엑스,https://www.inven.co.kr/webzine/news/?news=315341
5,"[LCK 톡톡] '비디디' 곽보성, ""KT, 아직 발전할 가능성 높아 끝까지 좋은...",http://www.osen.co.kr/article/G1112780807
6,"[LCK] BFX, '빅라', '데이스타' 교체 기용 암시",https://www.inven.co.kr/webzine/news/?news=315343
7,[기업家] 한진그룹 ㅣ 물류·항공 위기 대응부터 신사업 확장까지…'변...,https://www.cbci.co.kr/news/articleView.html?i...
8,"[Game & Now] 스마일게이트·LCK·SOOP, 콘텐츠 전방위 확장",https://www.ebn.co.kr/news/articleView.html?id...
9,"[LCK] 속도의 kt, 한진 꺾고 9년 만에 개막 4연승",http://www.dailyesports.com/view.php?ud=202604...
